# Exercise 12.2: Measure the multithreading speed-up

From *Programming in High Energy Particle Physics*, Chapter 12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch12/ex12_2_solution.ipynb)

Exercise 12.2 Measure the multithreading speed-up Benchmark the performance of RDataFrame with implicit multi-threading enabled vs. disabled. Process 1 million events and measure the speedup factor as a function of the number of threads (1, 2, 4, 8).

<details><summary>Hint</summary>

Use Listing 12.2 with ROOT::EnableImplicitMT(n) and time the event loop with TStopwatch or std::chrono . Run each configuration several times and discard the first run, which includes jitting and filling the disk cache. Fit Eq. (12.1) to your measured \(S(N)\) to extract the parallel fraction \(p\). Note that RDataFrame splits work by clusters of entries, so a small file with only a few clusters cannot keep many threads busy.

</details>

**Colab setup:** ROOT installation takes about 3–5 minutes and restarts the runtime once. After the restart, rerun the notebook from the top; the setup cells safely detect an existing mamba/ROOT installation.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import shutil
    if not shutil.which("mamba"):
        get_ipython().system("pip install -q condacolab")
        import condacolab
        condacolab.install()  # restarts runtime once

In [ ]:
if IN_COLAB:
    try:
        import ROOT
    except ImportError:
        get_ipython().system("mamba install -q -y -c conda-forge root")
        import ROOT

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy scipy matplotlib")

**Data:** one million deterministic synthetic entries. The event-loop timing excludes ROOT installation and discards the first run at each thread setting.

### Step 1: Build a one-million-event RDataFrame workload

In [ ]:
import ROOT, numpy as np
N=1_000_000
def work(n):
    ROOT.DisableImplicitMT()
    if n>1: ROOT.EnableImplicitMT(n)
    df=ROOT.RDataFrame(N).Define('pt','20.0+double((rdfentry_*37)%100)').Define('eta','double(int((rdfentry_*13)%50)-25)/10.0')
    return int(df.Filter('pt>40 && abs(eta)<2.4').Count().GetValue())
assert work(1)>0

### Step 2: Time 1, 2, 4 and 8 threads

In [ ]:
import time
from statistics import median
threads=np.array([1,2,4,8]); seconds=[]; counts=[]
for n in threads:
    work(int(n))  # warm up JIT and file/cache effects
    runs=[]
    for _ in range(3):
        t=time.perf_counter(); count=work(int(n)); runs.append(time.perf_counter()-t)
    counts.append(count); seconds.append(median(runs))
assert len(set(counts))==1
seconds=np.array(seconds); speedup=seconds[0]/seconds
print('Threads:',threads.tolist(),'seconds:',np.round(seconds,3).tolist(),'speedup:',np.round(speedup,2).tolist())

### Step 3: Fit Amdahl’s parallel fraction

In [ ]:
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
def amdahl(n,p): return 1/((1-p)+p/n)
pfit,_=curve_fit(amdahl,threads,speedup,p0=[.7],bounds=(0,1))
print('Fitted parallel fraction:',round(float(pfit[0]),3))
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(threads,speedup,'o',label='measured')
xx=np.linspace(1,8,100); ax.plot(xx,amdahl(xx,pfit[0]),label='Amdahl fit')
ax.plot(xx,amdahl(xx,.95),'--',label='reference p=0.95')
ax.set(xlabel='Number of threads',ylabel='Speed-up vs one thread',xticks=threads); ax.legend(); plt.show()

For p = 0.95, Amdahl’s law gives S(2)=1.90, S(4)=3.48 and S(8)=5.93. The measured curve may be lower or noisy because this short synthetic workload has fixed overhead and depends on Colab CPU allocation.

**Book answer:** Expect a sub-linear curve: with \(p = 0.95\), Eq. (12.1) gives \(S = 1.9,\ 3.5,\ 5.9\) for 2, 4 and 8 threads. For a light analysis on 1 million events, fixed costs (jitting, file opening) are a significant part of the run time, so measured speed-ups are often lower than this.